# CeNN Learned Kernel Concepts — Attention Preservation Lab

This notebook tests **new constant-state CeNN attention concepts** against frozen SmolLM2 attention. Q/K/V/O and RoPE stay unchanged; only the attention kernel is learned.

Variants:
- `learned_softplus`: separate learned positive Q/K feature maps
- `learned_softmax`: bounded positive feature maps with learned scale
- `norm_softplus`: Q/K normalization + learned per-head temperature + positive features
- `norm_taylor2`: bounded-score deterministic Taylor-2 control
- `taylor2_learned`: deterministic Taylor-2 kernel plus a learned positive correction bank

The benchmark trains only these feature maps and evaluates forward attention, normalization, attention KL/top-k, gradient fidelity, state size and runtime.


In [ ]:
import importlib, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
print('Repository ready:', REPO_DIR)


## Stage A — screen the new concepts on hard layer 18

This is intentionally small enough for a first Colab run. We use the same frozen teacher data for every candidate.


In [ ]:
import subprocess, sys
from pathlib import Path

STAGE_A = REPO_DIR / 'result' / 'cenn-learned-concepts-stage-a'
cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_learned_kernels.py'),
    '--base-model', 'HuggingFaceTB/SmolLM2-135M',
    '--context-length', '128',
    '--layers', '18',
    '--variants', 'learned_softplus,learned_softmax,norm_softplus,norm_taylor2,taylor2_learned',
    '--feature-dims', '256,512',
    '--train-sequences', '6',
    '--eval-sequences', '2',
    '--train-steps', '100',
    '--lr', '0.001',
    '--gradient-check',
    '--save-checkpoints',
    '--output-dir', str(STAGE_A),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

a = pd.read_csv(STAGE_A / 'learned_kernel_summary.csv').sort_values('selection_score', ascending=False)
display(a[[
    'variant','requested_feature_dim','effective_feature_dim','output_cosine','output_nmse',
    'partition_log_mae','attention_kl','topk_overlap','grad_mean_cosine','grad_mean_nmse',
    'state_vs_kv_ratio','break_even_tokens','runtime_ms','selection_score'
]])

labels = [f"{r.variant}\nF={int(r.effective_feature_dim)}" for _, r in a.iterrows()]
plt.figure(figsize=(13,5))
plt.bar(labels, a.output_cosine)
plt.axhline(0.80, linestyle='--', label='promising 0.80')
plt.axhline(0.90, linestyle=':', label='strong 0.90')
plt.xticks(rotation=45, ha='right')
plt.ylabel('attention-output cosine')
plt.title('Stage A: forward attention preservation')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,5))
plt.scatter(a.cenn_state_mib_fp32, a.output_cosine)
for _, r in a.iterrows():
    plt.annotate(f"{r.variant} F={int(r.effective_feature_dim)}", (r.cenn_state_mib_fp32, r.output_cosine), fontsize=8)
plt.xscale('log')
plt.xlabel('constant CeNN state MiB/layer (FP32)')
plt.ylabel('attention-output cosine')
plt.title('Quality / recurrent-state trade-off')
plt.tight_layout()
plt.show()


In [ ]:
# Select the two best distinct concepts for confirmation.
best_unique = a.drop_duplicates('variant').head(2)
TOP_VARIANTS = ','.join(best_unique.variant.tolist())
print('Top concepts for Stage B:', TOP_VARIANTS)

promising = a[(a.output_cosine >= 0.80) & (a.output_nmse <= 0.30) & (a.attention_kl <= 0.80)]
if len(promising):
    print('PROMISING candidates found:')
    display(promising)
else:
    print('No candidate has reached the first useful target yet; Stage B still tests whether more capacity/generalization helps.')


## Stage B — confirm the best concepts on easy / hard / late layers

Runs the best two Stage-A concepts on layers **0, 18, 29**, now with F=512/1024 where applicable and more optimization steps.


In [ ]:
STAGE_B = REPO_DIR / 'result' / 'cenn-learned-concepts-stage-b'
cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_learned_kernels.py'),
    '--base-model', 'HuggingFaceTB/SmolLM2-135M',
    '--context-length', '128',
    '--layers', '0,18,29',
    '--variants', TOP_VARIANTS,
    '--feature-dims', '512,1024',
    '--train-sequences', '8',
    '--eval-sequences', '3',
    '--train-steps', '160',
    '--lr', '0.0008',
    '--gradient-check',
    '--save-checkpoints',
    '--output-dir', str(STAGE_B),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
b = pd.read_csv(STAGE_B / 'learned_kernel_summary.csv').sort_values('selection_score', ascending=False)
display(b[[
    'layer','variant','requested_feature_dim','effective_feature_dim','output_cosine','output_nmse',
    'partition_log_mae','attention_kl','topk_overlap','grad_mean_cosine','grad_mean_nmse',
    'state_vs_kv_ratio','break_even_tokens','selection_score'
]])

pivot = b.pivot_table(index=['variant','effective_feature_dim'], columns='layer', values='output_cosine')
display(pivot)

for metric, title in [
    ('output_cosine', 'Forward attention cosine'),
    ('partition_log_mae', 'Partition log error (lower is better)'),
    ('grad_mean_cosine', 'Q/K/V gradient cosine'),
]:
    plt.figure(figsize=(9,5))
    for (variant, f), g in b.groupby(['variant','effective_feature_dim']):
        g = g.sort_values('layer')
        plt.plot(g.layer, g[metric], marker='o', label=f'{variant} F={int(f)}')
    plt.xlabel('Transformer layer')
    plt.ylabel(metric)
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## Decision criteria

A concept becomes worth integrating into a full CeNN LM when it reaches roughly:

| Level | Output cosine | NMSE | attention KL | gradient cosine |
|---|---:|---:|---:|---:|
| Promising | >=0.80 | <=0.30 | <=0.80 | >=0.70 |
| Strong | >=0.90 | <=0.15 | <=0.40 | >=0.80 |
| Near-equivalent | >=0.97 | <=0.05 | very low | high |

If learned features still saturate well below 0.8, the next architectural test should be **multiple recurrent kernel banks / mixtures**, not simply a larger F.


In [ ]:
# Optional: upload results/checkpoints to a private Hugging Face dataset/model repo if HF_TOKEN is available.
import os
token = os.environ.get('HF_TOKEN')
if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = 'vtava/TinyCeNN-LM-Colab-Backups'
    api.upload_folder(
        repo_id=repo_id,
        repo_type='model',
        folder_path=str(STAGE_B),
        path_in_repo='learned-kernel-concepts/latest',
        commit_message='Upload learned CeNN kernel concept benchmark',
    )
    print('Uploaded Stage B results to', repo_id)
else:
    print('HF_TOKEN not set; results remain in the Colab runtime:', STAGE_B)
